# 02 — Silver Transform | RHC Training Analytics

Camada Silver: padronização técnica, profiling, validação de chaves, integridade referencial e identificação de dados semiestruturados.

Fluxo: `Bronze Parquet → Data Quality → Silver Parquet`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, re
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)

BASE_DIR = Path('/content/drive/MyDrive/rhc-training-analytics/data')
BRONZE_ROOT = BASE_DIR / 'bronze'
SILVER_ROOT = BASE_DIR / 'silver'
RUN_TS = datetime.now(timezone.utc)
RUN_ID = RUN_TS.strftime('%Y%m%dT%H%M%SZ')

bronze_runs = sorted(BRONZE_ROOT.glob('extract_date=*/run_id=*'))
if not bronze_runs:
    raise FileNotFoundError('Nenhuma execução Bronze encontrada.')
BRONZE_DIR = bronze_runs[-1]
SILVER_DIR = SILVER_ROOT / f'process_date={RUN_TS:%Y-%m-%d}' / f'run_id={RUN_ID}'
SILVER_DIR.mkdir(parents=True, exist_ok=True)

print('Bronze selecionada:', BRONZE_DIR)
print('Silver destino:', SILVER_DIR)


In [ ]:
TABLES = [
    'profiles', 'body_measurements', 'workout_sessions', 'workout_exercises',
    'exercise_catalog', 'exercise_records', 'training_programs', 'program_phases',
    'program_sessions', 'program_exercises', 'program_enrollments',
    'program_exercise_exposures',
]
data = {t: pd.read_parquet(BRONZE_DIR / f'{t}.parquet') for t in TABLES}
pd.DataFrame([{'table': t, 'rows': len(df), 'columns': len(df.columns)} for t, df in data.items()])


## Profiling antes das transformações


In [ ]:
profile_records, table_quality = [], []
for table, df in data.items():
    table_quality.append({'table': table, 'rows': len(df), 'columns': len(df.columns), 'duplicate_rows': int(df.duplicated().sum())})
    for col in df.columns:
        s = df[col]
        profile_records.append({
            'table': table, 'column': col, 'dtype_before': str(s.dtype),
            'null_count': int(s.isna().sum()),
            'null_pct': round(float(s.isna().mean() * 100), 2),
            'distinct_count': int(s.nunique(dropna=True)),
        })
profile_before_df = pd.DataFrame(profile_records)
table_quality_df = pd.DataFrame(table_quality)
display(table_quality_df)
display(profile_before_df.sort_values('null_pct', ascending=False).head(40))


## Padronização técnica


In [ ]:
NULL_STRINGS = {'', 'null', 'none', 'nan', 'nat'}
DATE_RE = re.compile(r'(^|_)(date|created_at|updated_at|started_at|finished_at|completed_at|established_at|archived_at|measured_on|photo_date|last_date)$')
BOOL_TRUE = {'true', 't', '1', 'yes', 'y'}
BOOL_FALSE = {'false', 'f', '0', 'no', 'n'}

def clean_object_series(s):
    if s.dtype != 'object': return s
    out = s.map(lambda x: x.strip() if isinstance(x, str) else x)
    return out.map(lambda x: pd.NA if isinstance(x, str) and x.lower() in NULL_STRINGS else x)

def maybe_boolean(s):
    if s.dtype != 'object': return s
    vals = set(s.dropna().astype(str).str.lower().unique())
    if vals and vals.issubset(BOOL_TRUE | BOOL_FALSE):
        mapping = {**{v: True for v in BOOL_TRUE}, **{v: False for v in BOOL_FALSE}}
        return s.map(lambda x: mapping.get(str(x).lower()) if pd.notna(x) else pd.NA).astype('boolean')
    return s

def transform_table(df):
    out = df.copy()
    out.columns = [c.strip().lower() for c in out.columns]
    for col in out.columns:
        out[col] = clean_object_series(out[col])
        out[col] = maybe_boolean(out[col])
        if DATE_RE.search(col):
            parsed = pd.to_datetime(out[col], errors='coerce', utc=True)
            if out[col].notna().sum() == 0 or parsed.notna().sum() > 0:
                out[col] = parsed
    return out

silver = {t: transform_table(df) for t, df in data.items()}


## Validação de chaves


In [ ]:
key_checks = []
for table, df in silver.items():
    if 'id' in df.columns:
        key_checks.append({
            'table': table, 'key': 'id',
            'null_keys': int(df['id'].isna().sum()),
            'duplicate_keys': int(df['id'].duplicated().sum()),
            'unique_keys': int(df['id'].nunique(dropna=True)),
        })
# exercise_records usa chave lógica composta
er = silver['exercise_records']
key_checks.append({
    'table': 'exercise_records', 'key': 'profile_id+exercise_id',
    'null_keys': int(er[['profile_id','exercise_id']].isna().any(axis=1).sum()),
    'duplicate_keys': int(er.duplicated(['profile_id','exercise_id']).sum()),
    'unique_keys': int(er[['profile_id','exercise_id']].drop_duplicates().shape[0]),
})
key_checks_df = pd.DataFrame(key_checks)
display(key_checks_df)


## Integridade referencial


In [ ]:
RELATIONSHIPS = [
    ('body_measurements', 'profile_id', 'profiles', 'id'),
    ('workout_sessions', 'profile_id', 'profiles', 'id'),
    ('workout_exercises', 'session_id', 'workout_sessions', 'id'),
    ('workout_exercises', 'exercise_id', 'exercise_catalog', 'id'),
    ('exercise_records', 'profile_id', 'profiles', 'id'),
    ('exercise_records', 'exercise_id', 'exercise_catalog', 'id'),
    ('program_phases', 'program_id', 'training_programs', 'id'),
    ('program_sessions', 'program_id', 'training_programs', 'id'),
    ('program_exercises', 'session_id', 'program_sessions', 'id'),
    ('program_exercises', 'program_id', 'training_programs', 'id'),
    ('program_exercises', 'exercise_id', 'exercise_catalog', 'id'),
    ('program_enrollments', 'program_id', 'training_programs', 'id'),
    ('program_enrollments', 'profile_id', 'profiles', 'id'),
    ('program_exercise_exposures', 'profile_id', 'profiles', 'id'),
    ('program_exercise_exposures', 'enrollment_id', 'program_enrollments', 'id'),
    ('program_exercise_exposures', 'program_exercise_id', 'program_exercises', 'id'),
    ('program_exercise_exposures', 'program_id', 'training_programs', 'id'),
    ('program_exercise_exposures', 'workout_session_id', 'workout_sessions', 'id'),
    ('program_exercise_exposures', 'variation_exercise_id', 'exercise_catalog', 'id'),
]

relationship_checks = []
for child, fk, parent, pk in RELATIONSHIPS:
    if fk not in silver[child].columns or pk not in silver[parent].columns:
        relationship_checks.append({'child_table': child, 'foreign_key': fk, 'parent_table': parent, 'status': 'column_not_found', 'orphan_rows': None})
        continue
    values = silver[child][fk].dropna()
    parents = set(silver[parent][pk].dropna())
    relationship_checks.append({
        'child_table': child, 'foreign_key': fk, 'parent_table': parent,
        'status': 'checked', 'orphan_rows': int((~values.isin(parents)).sum()),
    })
relationship_checks_df = pd.DataFrame(relationship_checks)
display(relationship_checks_df)


## Colunas JSON / semiestruturadas


In [ ]:
def looks_like_json(value):
    if not isinstance(value, str): return False
    value = value.strip()
    if not (value.startswith('{') or value.startswith('[')): return False
    try:
        json.loads(value); return True
    except Exception:
        return False

json_columns = []
for table, df in silver.items():
    for col in df.select_dtypes(include='object').columns:
        non_null = df[col].dropna()
        if len(non_null) and non_null.head(50).map(looks_like_json).mean() >= .8:
            json_columns.append({'table': table, 'column': col, 'sample_count': min(len(non_null), 50)})
json_columns_df = pd.DataFrame(json_columns)
display(json_columns_df)


## Persistir Silver e auditorias


In [ ]:
for table, df in silver.items():
    df.to_parquet(SILVER_DIR / f'{table}.parquet', index=False)
profile_before_df.to_parquet(SILVER_DIR / '_profile_before.parquet', index=False)
table_quality_df.to_parquet(SILVER_DIR / '_table_quality.parquet', index=False)
key_checks_df.to_parquet(SILVER_DIR / '_key_checks.parquet', index=False)
relationship_checks_df.to_parquet(SILVER_DIR / '_relationship_checks.parquet', index=False)
json_columns_df.to_parquet(SILVER_DIR / '_json_columns.parquet', index=False)

manifest = {
    'project': 'rhc-training-analytics', 'layer': 'silver', 'run_id': RUN_ID,
    'processed_at_utc': RUN_TS.isoformat(), 'bronze_source': str(BRONZE_DIR),
    'silver_output': str(SILVER_DIR), 'tables': TABLES,
}
with (SILVER_DIR / '_manifest.json').open('w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)


## Critérios de sucesso


In [ ]:
assert all(len(data[t]) == len(silver[t]) for t in TABLES), 'Granularidade alterada na Silver.'
bad_keys = key_checks_df[(key_checks_df.null_keys > 0) | (key_checks_df.duplicate_keys > 0)]
missing_rel = relationship_checks_df[relationship_checks_df.status != 'checked']
orphans = relationship_checks_df[(relationship_checks_df.status == 'checked') & (relationship_checks_df.orphan_rows.fillna(0) > 0)]

print(f'✅ Silver processada: {len(TABLES)} tabelas.')
print(f'Problemas em chaves candidatas: {len(bad_keys)}')
print(f'Relacionamentos não validados por coluna ausente: {len(missing_rel)}')
print(f'Relacionamentos com registros órfãos: {len(orphans)}')
print(f'Colunas JSON detectadas: {len(json_columns_df)}')
display(bad_keys)
display(missing_rel)
display(orphans)
display(json_columns_df)
